# Проект: Дашборд конверсий


### Шаг 1: Подготовка к работе с данными
##### Мы создали Jypyter Notebook и подключаем библиотеки:

In [1]:
import pandas as pd
import numpy as np

##### Скачали и считываем два файла .csv Визитов и регистраций:

In [2]:
try:
    df1 = pd.read_csv('./regs_1k.csv')
    df2 = pd.read_csv('./visits_1k.csv')
except FileNotFoundError:
    df1 = pd.DataFrame(columns=['date', 'user_id', 'email', 'platform', 'registration_type'])
    df2 = pd.DataFrame(columns=['uuid', 'platform', 'user_agent', 'date'])

##### Для удобного отображения чисел, приводим к формату округления до целого числа:

In [3]:
pd.set_option('display.float_format', lambda x: '%.0f' % x)

##### Завершая шаг, сделаем предварительный анализ с помощью dataframe.describe для Визитов и Регистраций:

In [4]:
df1.describe(include='all')

,date,user_id,email,platform,registration_type
count,1000,1000,1000,1000,1000
unique,1000,NaN,997,3,4
top,2023-03-01T00:25:39,NaN,zanderson@example.org,android,email
freq,1,NaN,2,517,446
mean,NaN,4488623,NaN,NaN,NaN
std,NaN,2620568,NaN,NaN,NaN
min,NaN,22368,NaN,NaN,NaN
25%,NaN,2235489,NaN,NaN,NaN
50%,NaN,4473044,NaN,NaN,NaN
75%,NaN,6779707,NaN,NaN,NaN


In [5]:
df2.describe(include='all')

,uuid,platform,user_agent,date
count,1000,1000,1000,1000
unique,519,3,28,996
top,251a0926-ece3-4d77-aa42-ab569fdf9fe2,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-03-01T08:01:45
freq,4,954,71,2


### Шаг 2: Запросы к API
##### Для получения данных из API используем библиотеку requests, импортируем ее:

In [6]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()

True

##### Запрашиваем данные по посещениям и регистрациям по API за период 2023-03-01 -> 2023-09-01 и приводим из JSON в двумерную таблицу:

In [7]:
API_URL = os.getenv("API_URL")
DATE_BEGIN = os.getenv("DATE_BEGIN")
DATE_END = os.getenv("DATE_END")

In [8]:
params = {"begin": DATE_BEGIN, "end": DATE_END}
response_visits = requests.get(f"{API_URL}/visits", params=params, timeout=10)
df_api_visits = pd.DataFrame(response_visits.json())

In [9]:
response_regs = requests.get(f"{API_URL}/registrations", params=params, timeout=10)
df_api_regs = pd.DataFrame(response_regs.json())

### Шаг 3: Расчет метрик
##### Для начала приводим даты в единый формат:

In [10]:
df_api_visits["datetime"] = pd.to_datetime(df_api_visits["datetime"])
df_api_regs["datetime"] = pd.to_datetime(df_api_regs["datetime"])

##### Очистка визитов: убираем ботов из user_agent и исключаем платформу 'bot':

In [11]:
df_visits_clean = df_api_visits[
    (~df_api_visits["user_agent"].str.contains("bot", case=False, na=False))
    & (df_api_visits["platform"] != "bot")
].copy()

##### Сортируем таблицу по дате от ранних к более поздним и оставляем последний визит:

In [12]:
df_visits_clean = df_visits_clean.sort_values('datetime')
df_visits_clean = df_visits_clean.drop_duplicates(subset='visit_id', keep='last')

##### Выделение чистой даты для группировки:

In [13]:
df_visits_clean["date_group"] = df_visits_clean["datetime"].dt.strftime('%Y-%m-%d')
df_regs_clean = df_api_regs.copy()
df_regs_clean["date_group"] = df_regs_clean["datetime"].dt.strftime('%Y-%m-%d')

##### Агрегация данных по дате и платформе:

In [14]:
visits_grouped = df_visits_clean.groupby(["date_group", "platform"]).size().reset_index(name="visits")
regs_grouped = df_regs_clean.groupby(["date_group", "platform"]).size().reset_index(name="registrations")

##### Объединение таблиц визитов и регистраций:

In [15]:
final_df = pd.merge(visits_grouped, regs_grouped, on=["date_group", "platform"], how="outer").reset_index(drop=True)

##### Заполнение пустых значений нулями:

In [16]:
final_df["visits"] = final_df["visits"].fillna(0).astype(int)
final_df["registrations"] = final_df["registrations"].fillna(0).astype(int)

##### Расчет конверсии в процентах:

In [17]:
final_df["conversion"] = np.where(final_df["visits"] > 0, (final_df["registrations"] / final_df["visits"]) * 100, 0.0,)

##### Сортировка по дате от ранних к поздним:


In [18]:
final_df = final_df.sort_values("date_group")

##### Сохранение итогового результата в conversion.json:

In [19]:
import json
final_df_json = final_df.copy()
if final_df_json["date_group"].dtype == 'int64':
    final_df_json["date_group"] = pd.to_datetime(final_df_json["date_group"], unit='ms').dt.strftime('%Y-%m-%d').astype(str)
else:
    final_df_json["date_group"] = pd.to_datetime(final_df_json["date_group"]).dt.strftime('%Y-%m-%d').astype(str)
final_df_json["visits"] = final_df_json["visits"].astype(int)
final_df_json["registrations"] = final_df_json["registrations"].astype(int)
final_df_json["conversion"] = final_df_json["conversion"].astype(float)
data_dict_conv = final_df_json.to_dict(orient="dict")
with open("./conversion.json", "w", encoding="utf-8") as f:
    json.dump(data_dict_conv, f, ensure_ascii=False)

### Шаг 4: Добавление рекламы

##### Считываем файл ads.csv: 

In [20]:
df_ads = pd.read_csv('./ads.csv')

##### Приведение дат к единому типу и формату:

In [21]:
df_ads["date_group"] = pd.to_datetime(df_ads["date"]).dt.strftime('%Y-%m-%d')

##### Подсчет визитов и регистраций по дням:

In [22]:
visits_by_date = df_visits_clean.groupby('date_group').size().reset_index(name='visits')
regs_by_date = df_regs_clean.groupby('date_group').size().reset_index(name='registrations')

##### Агрегация рекламных расходов по дням:

In [23]:
ads_grouped = df_ads.groupby(['date_group', 'utm_campaign'])['cost'].sum().reset_index()

##### Создание базовой таблицы:

In [24]:
df_base = pd.merge(visits_by_date, regs_by_date, on='date_group', how='outer')

##### Объединение продуктовых метрик с рекламой:

In [25]:
df_ads_merged = pd.merge(df_base, ads_grouped, on='date_group', how='outer')

##### чистка пустых значений и замена их на 0 и none:

In [26]:
df_ads_merged['visits'] = df_ads_merged['visits'].fillna(0).astype(int)
df_ads_merged['registrations'] = df_ads_merged['registrations'].fillna(0).astype(int)
df_ads_merged['cost'] = df_ads_merged['cost'].fillna(0).astype(int)
df_ads_merged['utm_campaign'] = df_ads_merged['utm_campaign'].fillna('none')

##### Сортировка и фильтрация столбцов:

In [27]:
df_ads_final = df_ads_merged.sort_values(by='date_group').reset_index(drop=True)
final_columns = ['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']
df_ads_final = df_ads_final[final_columns]

##### Сохраниние файла в .json:

In [28]:
df_ads_json = df_ads_final.copy()
if df_ads_json["date_group"].dtype == 'int64':
    df_ads_json["date_group"] = pd.to_datetime(df_ads_json["date_group"], unit='ms').dt.strftime('%Y-%m-%d').astype(str)
else:
    df_ads_json["date_group"] = pd.to_datetime(df_ads_json["date_group"]).dt.strftime('%Y-%m-%d').astype(str)
df_ads_json["visits"] = df_ads_json["visits"].astype(int)
df_ads_json["registrations"] = df_ads_json["registrations"].astype(int)
df_ads_json["cost"] = df_ads_json["cost"].astype(int)
data_dict_ads = df_ads_json.to_dict(orient="dict")
with open("./ads.json", "w", encoding="utf-8") as f:
    json.dump(data_dict_ads, f, ensure_ascii=False)

### Шаг 4: Визуализация

##### Импортируем необходимые библиотеки и создадим директоию charts:

In [29]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [30]:
os.makedirs('./charts', exist_ok=True)

##### Включаем сетку и настраиваем базовый размер:

In [31]:
plt.style.use('default')
plt.rcParams['font.size'] = 9

##### Подготовим копию данных с понятными датами на оси X:

In [32]:
df_plot = df_ads_final.copy()
df_plot = df_plot[(df_plot['visits'] > 0) | (df_plot['registrations'] > 0)]
df_plot['date_str'] = pd.to_datetime(df_plot['date_group'], unit='ms').dt.strftime('%Y-%m-%d')

##### График 1: Итоговые визиты:

In [33]:
fig, ax = plt.subplots(figsize=(20, 7), dpi=100)
ax.grid(axis='y', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
bars = ax.bar(df_plot['date_str'], df_plot['visits'], color='#72bcd4', zorder=3, width=0.7)
ax.bar_label(bars, padding=3, fontsize=8, rotation=0)
ax.margins(y=0.15)
plt.title('Total Visits', fontsize=14, pad=15)
plt.ylabel('visits')
plt.xlabel('date_group')
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.savefig('./charts/total_visits.png', bbox_inches='tight', dpi=100)
plt.close()


##### График 2: Итоговые регистрации:

In [34]:
fig, ax = plt.subplots(figsize=(20, 7), dpi=100)
ax.grid(axis='y', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
bars = ax.bar(df_plot['date_str'], df_plot['registrations'], color='#72bcd4', zorder=3, width=0.7)
ax.bar_label(bars, padding=3, fontsize=8, rotation=0)
ax.margins(y=0.15)
plt.title('Total Registrations', fontsize=14, pad=15)
plt.ylabel('registrations')
plt.xlabel('date_group')
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.savefig('./charts/total_registrations.png', bbox_inches='tight', dpi=100)
plt.close()


##### Подготовка данных по платформам:

In [35]:
df_conv_plot = final_df.copy()
df_conv_plot = df_conv_plot[(df_conv_plot['visits'] > 0) | (df_conv_plot['registrations'] > 0)]
df_conv_plot['date_str'] = pd.to_datetime(df_conv_plot['date_group'], unit='ms').dt.strftime('%Y-%m-%d')

df_visits_stacked = df_conv_plot.pivot(index='date_str', columns='platform', values='visits').fillna(0)
df_regs_stacked = df_conv_plot.pivot(index='date_str', columns='platform', values='registrations').fillna(0)

platform_order = ['android', 'ios', 'web']
df_visits_stacked = df_visits_stacked.reindex(columns=platform_order)
df_regs_stacked = df_regs_stacked.reindex(columns=platform_order)

colors = ['#4c72b0', '#dd8452', '#55a868']

##### График 3: Итоговые визиты с разбивкой по платформам: web, android, ios:

In [36]:
fig, ax = plt.subplots(figsize=(20, 7), dpi=100)
ax.grid(axis='both', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
df_visits_stacked.plot(kind='bar', stacked=True, color=colors, width=0.7, ax=ax, zorder=3)
plt.title('Visits by Platform (Stacked)', fontsize=14, pad=15)
plt.ylabel('visits')
plt.xlabel('date_group')
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend(title='platform', loc='upper right')
plt.savefig('./charts/visits_by_platform.png', bbox_inches='tight', dpi=100)
plt.close()

##### График 4: Итоговые регистрации с разбивкой по платформам: web, android,ios

In [37]:
fig, ax = plt.subplots(figsize=(20, 7), dpi=100)
ax.grid(axis='both', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
df_regs_stacked.plot(kind='bar', stacked=True, color=colors, width=0.7, ax=ax, zorder=3)
plt.title('Registrations by Platform (Stacked)', fontsize=14, pad=15)
plt.ylabel('registrations')
plt.xlabel('date_group')
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend(title='platform', loc='upper right')
plt.savefig('./charts/registrations_by_platform.png', bbox_inches='tight', dpi=100)
plt.close()

##### График 5:Конверсия по каждой платформе и средняя конверсия:

In [38]:
mean_conv = df_conv_plot['conversion'].mean()
fig, ax = plt.subplots(figsize=(20, 7), dpi=100)
ax.grid(axis='y', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
for plat in platform_order:
    df_plat = df_conv_plot[df_conv_plot['platform'] == plat].sort_values('date_str')
    ax.plot(df_plat['date_str'], df_plat['conversion'], marker='o', linewidth=1.5, label=plat, zorder=3)
ax.axhline(mean_conv, color='red', linestyle='--', linewidth=1.2, label=f'Mean Conversion ({mean_conv:.2f}%)', zorder=4)
plt.title('Conversion by Platform vs Mean Conversion', fontsize=14, pad=15)
plt.xlabel('date_group')
plt.ylabel('Conversion (%)')
ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend()
plt.savefig('./charts/conversion_by_platform.png', bbox_inches='tight', dpi=100)
plt.close()


##### График 6: Стоимости реклам:

In [39]:
fig, ax = plt.subplots(figsize=(20, 7), dpi=100)
ax.grid(axis='both', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
ax.plot(df_plot['date_str'], df_plot['cost'], color='#4c72b0', marker='o', linewidth=1.5, zorder=3)
for x_val, y_val in zip(df_plot['date_str'], df_plot['cost']):
    if y_val > 0:
        ax.annotate(
            f'{int(y_val)} RUB',
            (x_val, y_val),
            ha='center',
            va='bottom',
            fontsize=8,
            xytext=(0, 5),
            textcoords='offset points'
        )
ax.margins(y=0.15)
plt.title('Aggregated Ad Campaign Costs (by day)', fontsize=14, pad=15)
plt.ylabel('Cost (RUB)', fontsize=11)
plt.xlabel('Date', fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.savefig('./charts/advertising_costs.png', bbox_inches='tight', dpi=100)
plt.close()

##### График 7:Визиты и регистрации за весь период с цветовым выделением рекламной кампании:

##### Подготовка данных:

In [40]:
import matplotlib.patches as mpatches
campaign_intervals = []
unique_camps = [c for c in df_plot['utm_campaign'].unique() if c != 'none']
for camp in unique_camps:
    df_camp_dates = df_plot[df_plot['utm_campaign'] == camp]
    if not df_camp_dates.empty:
        start_idx = df_camp_dates.index.min()
        end_idx = df_camp_dates.index.max()
        campaign_intervals.append({
            'campaign': camp,
            'start_idx': start_idx,
            'end_idx': end_idx
        })
hexlet_palette = ['#b0c4de', '#f4a460', '#98fb98', '#afeeee']
camp_colors = {camp: hexlet_palette[i % len(hexlet_palette)] for i, camp in enumerate(unique_camps)}

def draw_hexlet_spans(ax):
    for interval in campaign_intervals:
        ax.axvspan(
            interval['start_idx'], 
            interval['end_idx'], 
            color=camp_colors[interval['campaign']], 
            alpha=0.6, 
            zorder=1
        )
def add_hexlet_legend(ax, main_line, mean_line):
    handles = [main_line, mean_line]
    for camp, col in camp_colors.items():
        handles.append(mpatches.Patch(color=col, alpha=0.6, label=camp))
    ax.legend(handles=handles, loc='lower left', fontsize=8)

##### Визиты за весь период с цветовым выделением рекламной кампании:

In [41]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=100)
ax.grid(axis='both', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
draw_hexlet_spans(ax)
line_v, = ax.plot(
    df_plot['date_str'], df_plot['visits'], 
    color='black', marker='o', markersize=4, linewidth=1, 
    label='Visits', zorder=3
)
mean_visits = df_plot['visits'].mean()
line_mean_v = ax.axhline(
    mean_visits, 
    color='gray', linestyle='--', linewidth=1, 
    label='Average Number of Visits', zorder=2
)
plt.title('Visits during marketing active days', fontsize=12, pad=15)
plt.ylabel('Unique Visits', fontsize=10)
ax.xaxis.set_major_locator(ticker.MultipleLocator(7))
plt.xticks(rotation=45, ha='right', fontsize=9)
add_hexlet_legend(ax, line_v, line_mean_v)
plt.savefig('./charts/visits_by_campaign.png', bbox_inches='tight', dpi=100)
plt.close()

##### Регистрации за весь период с цветовым выделением рекламной кампании: 

In [42]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=100)
ax.grid(axis='both', linestyle='-', linewidth=0.5, color='lightgray', zorder=0)
draw_hexlet_spans(ax)
line_r, = ax.plot(
    df_plot['date_str'], df_plot['registrations'], 
    color='#228b22', marker='o', markersize=4, linewidth=1, 
    label='Registrations', zorder=3
)
mean_regs = df_plot['registrations'].mean()
line_mean_r = ax.axhline(
    mean_regs, 
    color='gray', linestyle='--', linewidth=1, 
    label='Average Number of Registration', zorder=2
)
plt.title('Registrations during marketing active days', fontsize=12, pad=15)
plt.ylabel('Unique Users', fontsize=10)
ax.xaxis.set_major_locator(ticker.MultipleLocator(7))
plt.xticks(rotation=45, ha='right', fontsize=9)
add_hexlet_legend(ax, line_r, line_mean_r)
plt.savefig('./charts/registrations_by_campaign.png', bbox_inches='tight', dpi=100)
plt.close()